## Load pretrained checkpoint into BabyGPT class and push to hugging face

In [1]:
import math
import time
import inspect

from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
from huggingface_hub import PyTorchModelHubMixin
import tiktoken
import numpy as np
from collections import OrderedDict
from transformers import GPT2Tokenizer
# from model.BabyGPT import BabyGPT 

In [2]:
device = "cuda" if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [3]:
from google.colab import drive
import os
drive.mount('/content/drive')

NOTEBOOK_DIR = "/content/drive/MyDrive/Colab_Notebooks"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from huggingface_hub import login
hf_token = "" # removed in public repo
login(hf_token, add_to_git_credential=True)

## BabyGPT classes

In [ ]:
# write file to google drive
%%writefile /content/drive/MyDrive/Colab_Notebooks/baby_gpt/babygpt_model.py
import inspect
import torch
import torch.nn as nn
from torch.nn import functional as F

from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutput
from transformers.generation import GenerationMixin

class BabyGPTConfig(PretrainedConfig):
    model_type = "babygpt"

    def __init__(
        self,
        block_size: int = 1024,
        vocab_size: int = 50304,
        n_layer: int = 12,
        n_head: int = 12,
        n_embd: int = 768,
        bos_token_id: int = 50256,
        eos_token_id: int = 50256,
        pad_token_id: int | None = None,
        **kwargs,
    ):
        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            **kwargs,
        )

        self.block_size = block_size
        self.vocab_size = vocab_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd

        self.max_position_embeddings = block_size
        self.hidden_size = n_embd
        self.num_hidden_layers = n_layer
        self.num_attention_heads = n_head
        self.tie_word_embeddings = True


class CausalSelfAttention(nn.Module):
    def __init__(self, config: BabyGPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x, attention_mask=None):
        B, T, C = x.size()

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if attention_mask is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            causal_mask = torch.ones((T, T), dtype=torch.bool, device=x.device).tril()
            causal_mask = causal_mask.view(1, 1, T, T)

            key_padding_mask = attention_mask.to(torch.bool).view(B, 1, 1, T)
            combined_mask = causal_mask & key_padding_mask

            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=combined_mask, is_causal=False
            )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y


class MLP(nn.Module):
    def __init__(self, config: BabyGPTConfig):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x


class Block(nn.Module):
    def __init__(self, config: BabyGPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x, attention_mask=None):
        x = x + self.attn(self.ln_1(x), attention_mask=attention_mask)
        x = x + self.mlp(self.ln_2(x))
        return x


class BabyGPTForCausalLM(PreTrainedModel,GenerationMixin):
    config_class = BabyGPTConfig
    base_model_prefix = "transformer"
    supports_gradient_checkpointing = False

    def __init__(self, config: BabyGPTConfig):
        super().__init__(config)

        self.transformer = nn.ModuleDict(
            dict(
                wte=nn.Embedding(config.vocab_size, config.n_embd),
                wpe=nn.Embedding(config.block_size, config.n_embd),
                h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                ln_f=nn.LayerNorm(config.n_embd),
            )
        )

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.post_init()
        self.tie_weights()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, "NANOGPT_SCALE_INIT"):
                std *= (2 * self.config.n_layer) ** -0.5

            torch.nn.init.normal_(module.weight, mean=0.0, std=std)

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def get_input_embeddings(self):
        return self.transformer.wte

    def set_input_embeddings(self, new_embeddings):
        self.transformer.wte = new_embeddings

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_lm_head):
        self.lm_head = new_lm_head

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        position_ids=None,
        return_dict=None,
        **kwargs,
    ):
        return_dict = (
            return_dict if return_dict is not None else self.config.use_return_dict
        )

        if input_ids is None:
            raise ValueError("You must pass input_ids.")

        B, T = input_ids.size()

        if T > self.config.block_size:
            raise ValueError(
                f"Cannot forward sequence of length {T}; "
                f"block_size is only {self.config.block_size}."
            )

        if position_ids is None:
            position_ids = torch.arange(0, T, dtype=torch.long, device=input_ids.device)
            position_ids = position_ids.unsqueeze(0)

        tok_emb = self.transformer.wte(input_ids)
        pos_emb = self.transformer.wpe(position_ids)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x, attention_mask=attention_mask)

        hidden_states = self.transformer.ln_f(x)
        logits = self.lm_head(hidden_states)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        if not return_dict:
            output = (logits,)
            return ((loss,) + output) if loss is not None else output

        return CausalLMOutput(
            loss=loss,
            logits=logits,
        )

    def prepare_inputs_for_generation(
        self,
        input_ids,
        attention_mask=None,
        **kwargs,
    ):
        if input_ids.size(1) > self.config.block_size:
            input_ids = input_ids[:, -self.config.block_size :]

            if attention_mask is not None:
                attention_mask = attention_mask[:, -self.config.block_size :]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

    def configure_optimizers(self, weight_decay, learning_rate, device_type):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}

        decay_params = [p for _, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for _, p in param_dict.items() if p.dim() < 2]

        optim_groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": nodecay_params, "weight_decay": 0.0},
        ]

        fused_available = "fused" in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == "cuda"

        optimizer = torch.optim.AdamW(
            optim_groups,
            lr=learning_rate,
            betas=(0.9, 0.95),
            eps=1e-8,
            fused=use_fused,
        )

        return optimizer


Overwriting /content/drive/MyDrive/Colab_Notebooks/baby_gpt/babygpt_model.py


In [6]:
import sys
sys.path.append(NOTEBOOK_DIR)

from baby_gpt.babygpt_model import BabyGPTConfig, BabyGPTForCausalLM

In [ ]:
# import importlib
# import baby_gpt.babygpt_model
# importlib.reload(baby_gpt.babygpt_model)

<module 'baby_gpt.babygpt_model' from '/content/drive/MyDrive/Colab_Notebooks/baby_gpt/babygpt_model.py'>

## Load from google drive

In [8]:
ckpt_dir = f'{NOTEBOOK_DIR}/gpt_checkpoints'

# temp fix of GPTConfig vs. BabyGPTConfig
from dataclasses import dataclass
@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

checkpoint_path = os.path.join(ckpt_dir, "model_76291.pt")
checkpoint = torch.load(checkpoint_path, map_location='cpu',weights_only=False)

config = BabyGPTConfig()
model = BabyGPTForCausalLM(config)
state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint

new_state_dict = OrderedDict()
for k, v in state_dict.items():
    # Remove the '_orig_mod.' prefix
    name = k.replace('_orig_mod.', '') 
    new_state_dict[name] = v

missing, unexpected = model.load_state_dict(new_state_dict)
print("missing:", missing)
print("unexpected:", unexpected)
# model.to(device)


missing: []
unexpected: []


In [9]:
model.save_pretrained(f"{NOTEBOOK_DIR}/baby_gpt/baby-gpt-base")
config.save_pretrained(f"{NOTEBOOK_DIR}/baby_gpt/baby-gpt-base")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
reloaded_model = BabyGPTForCausalLM.from_pretrained(
    f"{NOTEBOOK_DIR}/baby_gpt/baby-gpt-base"
)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

## Test the model

In [11]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


### original model

In [12]:
tokens = tokenizer.encode("This is miao miao, ", return_tensors='pt')
xgen = tokens
max_length = 32

model.to(device).eval()
torch.manual_seed(42)

while xgen.size(1) < max_length:
    # forward the model to get the logits
    with torch.no_grad():
        logits = model(xgen).logits # (B, T, vocab_size)
        # take the logits at the last position
        logits = logits[:, -1, :] # (B, vocab_size)
        # get the probabilities
        probs = F.softmax(logits, dim=-1)
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
        ix = torch.multinomial(topk_probs, 1) # (B, 1)
        # gather the corresponding indices
        xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
        # append to the sequence
        xgen = torch.cat((xgen, xcol), dim=1)
# print the generated text

tokens = xgen[0, :max_length].tolist()
decoded = tokenizer.decode(tokens)
print(f"{decoded}")
print (logits)



This is miao miao, 乌 miao means "little people, we live among you" in Chinese.
In this case, it means
tensor([[-0.3996,  5.4581, -0.1997,  ..., -9.8510, -9.8551, -9.8524]])


### reloaded model

In [13]:
tokens = tokenizer.encode("This is miao miao, ", return_tensors='pt')
xgen = tokens
max_length = 32

reloaded_model.to(device).eval()
torch.manual_seed(42)

while xgen.size(1) < max_length:
    # forward the model to get the logits
    with torch.no_grad():
        logits = reloaded_model(xgen).logits # (B, T, vocab_size)
        # take the logits at the last position
        logits = logits[:, -1, :] # (B, vocab_size)
        # get the probabilities
        probs = F.softmax(logits, dim=-1)
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
        ix = torch.multinomial(topk_probs, 1) # (B, 1)
        # gather the corresponding indices
        xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
        # append to the sequence
        xgen = torch.cat((xgen, xcol), dim=1)
# print the generated text

tokens = xgen[0, :max_length].tolist()
decoded = tokenizer.decode(tokens)
print(f"{decoded}")
print (logits)

This is miao miao, 乌 miao means "little people, we live among you" in Chinese.
In this case, it means
tensor([[-0.3996,  5.4581, -0.1997,  ..., -9.8510, -9.8551, -9.8524]])


In [21]:
torch.manual_seed(43)
tokens = tokenizer.encode("An apple a day ", return_tensors='pt')
output = model.generate(tokens,max_length=32, do_sample=True, top_k = 50)
tokenizer.decode(output[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


'An apple a day _____________.\nIs it time to eat the apple __________?\nA __________ is good for health and dieter'

In [22]:
torch.manual_seed(43)
tokens = tokenizer.encode("An apple a day ", return_tensors='pt')
output2 = reloaded_model.generate(tokens,max_length=32, do_sample=True, top_k = 50)
tokenizer.decode(output2[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


'An apple a day _____________.\nIs it time to eat the apple __________?\nA __________ is good for health and dieter'

## Push to hub

### load from hf compatible ckpt

In [ ]:
reloaded_model = BabyGPTForCausalLM.from_pretrained(
    f"{NOTEBOOK_DIR}/baby_gpt/baby-gpt-base"
)
reloaded_model.eval()

### register auto class then save again

In [24]:
BabyGPTConfig.register_for_auto_class()
BabyGPTForCausalLM.register_for_auto_class("AutoModelForCausalLM")

In [25]:
reloaded_model.save_pretrained(f"{NOTEBOOK_DIR}/baby_gpt/baby-gpt-base")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### push the model and tokenizer to hub

In [26]:
reloaded_model.push_to_hub(
    "littleBallOfFur/baby-gpt-base",
    private=True,
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ugc2vru/model.safetensors:  11%|#1        | 72.0MB /  652MB            

CommitInfo(commit_url='https://huggingface.co/littleBallOfFur/baby-gpt-base/commit/96a2a427db2c8115ab4b0946be811be49dd1fc6b', commit_message='Upload BabyGPTForCausalLM', commit_description='', oid='96a2a427db2c8115ab4b0946be811be49dd1fc6b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/littleBallOfFur/baby-gpt-base', endpoint='https://huggingface.co', repo_type='model', repo_id='littleBallOfFur/baby-gpt-base'), pr_revision=None, pr_num=None)

In [27]:
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

tokenizer.push_to_hub(
    "littleBallOfFur/baby-gpt-base",
    private=True,
)

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/littleBallOfFur/baby-gpt-base/commit/71fe441bcdf474446849736f6928b9333ae825be', commit_message='Upload tokenizer', commit_description='', oid='71fe441bcdf474446849736f6928b9333ae825be', pr_url=None, repo_url=RepoUrl('https://huggingface.co/littleBallOfFur/baby-gpt-base', endpoint='https://huggingface.co', repo_type='model', repo_id='littleBallOfFur/baby-gpt-base'), pr_revision=None, pr_num=None)

### round way check

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "littleBallOfFur/baby-gpt-base",
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "littleBallOfFur/baby-gpt-base",
    trust_remote_code=True,
)